# Experiment: ObviousBench Paper Figures

Objective:
- Rebuild the paper tables and figures from one declared result source.
- Preview the four current result figures without manually recreating charts.
- Keep placeholder proof-point data visibly separate from the future final paper sweep.

Success criteria:
- The asset builder completes from a clean kernel.
- `paper/figures/*.pdf` are regenerated deterministically.
- The notebook prints a compact result summary and optional local PNG previews.


In [ ]:
from __future__ import annotations

import csv
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "scripts/build_paper_assets.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts/build_paper_assets.py").exists(), ROOT

PYTHON = ROOT / ".venv/bin/python"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

MANIFEST = ROOT / "data/splits/paper_v1_manifest.jsonl"
HUMAN_BASELINE = ROOT / "data/human_baseline/paper_v1.csv"
PLACEHOLDER_RESULTS = ROOT / "docs/shareable/2026-05-31-obviousbench-proof-point"
FINAL_RESULTS = ROOT / "results/paper_v1_final"
TABLES_OUT = ROOT / "paper/tables"
FIGURES_OUT = ROOT / "paper/figures"
RENDERER = "chrome-svg"

RESULTS_DIR = FINAL_RESULTS if FINAL_RESULTS.exists() else PLACEHOLDER_RESULTS
RESULT_SOURCE_FLAG = (
    "--final-results-dir" if FINAL_RESULTS.exists() else "--placeholder-results-dir"
)

config = {
    "root": str(ROOT),
    "python": str(PYTHON),
    "manifest": str(MANIFEST.relative_to(ROOT)),
    "result_source": str(RESULTS_DIR.relative_to(ROOT)),
    "result_source_kind": RESULT_SOURCE_FLAG.removeprefix("--").replace("-", "_"),
    "renderer": RENDERER,
}
config

## Plan

- Use the repo asset builder as the single chart source of truth.
- Prefer the final paper sweep automatically when `results/paper_v1_final` exists.
- Fall back to the labeled proof-point placeholder bundle while drafting.
- Render with the Chrome-backed SVG path when available, with the pure-Python PDF path as fallback.


In [ ]:
cmd = [
    str(PYTHON),
    "scripts/build_paper_assets.py",
    "--manifest",
    str(MANIFEST),
    "--human-baseline",
    str(HUMAN_BASELINE),
    RESULT_SOURCE_FLAG,
    str(RESULTS_DIR),
    "--figure-renderer",
    RENDERER,
    "--out",
    str(TABLES_OUT),
    "--figures-out",
    str(FIGURES_OUT),
]

completed = subprocess.run(
    cmd,
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise SystemExit(completed.returncode)

## Compact Result Summary

This cell is intentionally standard-library only. It gives enough signal to spot accidental source changes before inspecting the PDFs.


In [ ]:
def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def first_existing(directory: Path, names: tuple[str, ...]) -> Path:
    for name in names:
        candidate = directory / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError((directory, names))


def metric(row: dict[str, str], *keys: str) -> float:
    for key in keys:
        value = (row.get(key) or "").strip()
        if value:
            parsed = float(value)
            return parsed / 100.0 if key.endswith("_pct") else parsed
    return 0.0


comparison_path = first_existing(
    RESULTS_DIR,
    ("model-comparison.csv", "comparison.csv", "leaderboard.csv"),
)
family_path = first_existing(
    RESULTS_DIR,
    ("family-comparison.csv", "family_comparison.csv", "family-heatmap.csv"),
)
comparison_rows = read_csv(comparison_path)
family_rows = read_csv(family_path)

top_rows = sorted(
    comparison_rows,
    key=lambda row: (
        -metric(row, "strict_accuracy", "strict_accuracy_pct", "accuracy_pct"),
        row.get("label", ""),
    ),
)[:8]

print(f"Comparison rows: {len(comparison_rows)} from {comparison_path.relative_to(ROOT)}")
print(f"Family rows: {len(family_rows)} from {family_path.relative_to(ROOT)}")
print("\nTop strict-accuracy rows:")
for row in top_rows:
    label = row.get("label") or row.get("model") or "unknown"
    strict = metric(row, "strict_accuracy", "strict_accuracy_pct", "accuracy_pct")
    answer = metric(row, "answer_accuracy", "answer_accuracy_pct", "accuracy_pct")
    cost = row.get("estimated_cost_usd") or row.get("cost_usd") or ""
    print(f"- {label}: strict={strict:.1%}, answer={answer:.1%}, cost={cost}")

## Figure Outputs

The paper currently expects these four result figures. The PDFs are the manuscript artifacts; PNG previews are optional local conveniences for notebook review.


In [ ]:
figure_paths = [
    FIGURES_OUT / "leaderboard.pdf",
    FIGURES_OUT / "family_heatmap.pdf",
    FIGURES_OUT / "answer_format_gap.pdf",
    FIGURES_OUT / "cost_frontier.pdf",
]

for path in figure_paths:
    print(f"{path.relative_to(ROOT)} - {path.stat().st_size:,} bytes")

In [ ]:
pdftoppm = shutil.which("pdftoppm")
preview_dir = ROOT / "tmp/paper-figure-previews"

if pdftoppm is None:
    print("pdftoppm is not installed; open the PDFs above for visual inspection.")
else:
    preview_dir.mkdir(parents=True, exist_ok=True)
    preview_paths = []
    for pdf_path in figure_paths:
        out_prefix = preview_dir / pdf_path.stem
        subprocess.run(
            [pdftoppm, "-png", "-singlefile", "-r", "150", str(pdf_path), str(out_prefix)],
            cwd=ROOT,
            check=True,
        )
        preview_paths.append(out_prefix.with_suffix(".png"))
    print(f"Wrote {len(preview_paths)} previews to {preview_dir.relative_to(ROOT)}")

    if importlib.util.find_spec("IPython"):
        from IPython.display import Image, display

        for preview_path in preview_paths:
            print(preview_path.name)
            display(Image(filename=str(preview_path)))

## Notes For The Paper Draft

- Do not manually edit generated figure PDFs.
- Treat placeholder proof-point figures as layout and critique aids only.
- When the final sweep lands, place it at `results/paper_v1_final` or pass `--final-results-dir` directly, then rerun this notebook and `make -C paper pdf`.
- If a figure looks weak, improve `obviousbench/research/paper_assets.py` and rerun this notebook so the chart logic stays versioned.
